`exp2/{tocabi,t1,kapex,g1}` 의 `footstep_eval_*.csv`를 로봇별로 모아,
`cmd_z ∈ [-0.15, 0.20]`만 0.05 m 구간으로 나눈 뒤
**foothold error** $|xy|=\sqrt{e_x^2+e_y^2}$ [m] 와 **yaw error** $|e_{\mathrm{yaw}}|$ [rad] 의 mean $\pm$ std 를 구합니다.
마지막에 논문 표 형식 LaTeX를 출력합니다 (셀 위: $|xy|$, 아래: $|yaw|$).


In [ ]:
from pathlib import Path
import csv
import numpy as np

HERE = Path("/home/yong/unitree_ws/g1_controller/log/exp2")
ROBOTS = ["g1", "t1", "tocabi", "kapex"]
DISPLAY = {"tocabi": "TOCABI", "t1": "Booster T1", "kapex": "KAPEX", "g1": "Unitree G1"}

EDGES = np.round(np.arange(-0.15, 0.20 + 0.05, 0.05), 3)
BINS = [f"{a:.2f}~{b:.2f}" for a, b in zip(EDGES[:-1], EDGES[1:])]


def bin_of(z):
    if z < EDGES[0] or z > EDGES[-1]:
        return None
    for i in range(len(EDGES) - 1):
        lo, hi = EDGES[i], EDGES[i + 1]
        if i == 0:
            if lo <= z <= hi:
                return BINS[i]
        elif lo < z <= hi:
            return BINS[i]
    return None


def load_robot(name):
    xy, yaw = {b: [] for b in BINS}, {b: [] for b in BINS}
    for path in sorted((HERE / name).glob("*.csv")):
        with path.open() as f:
            for row in csv.DictReader(f):
                z = float(row["cmd_z"])
                b = bin_of(z)
                if b is None:
                    continue
                xy[b].append(np.hypot(float(row["err_x"]), float(row["err_y"])))
                yaw[b].append(abs(float(row["err_yaw"])))
    return xy, yaw


def mean_std(vals):
    a = np.asarray(vals, dtype=float)
    if a.size == 0:
        return None
    if a.size == 1:
        return float(a[0]), float("nan")
    return float(a.mean()), float(a.std(ddof=1))


stats = {}
for robot in ROBOTS:
    xy, yaw = load_robot(robot)
    stats[robot] = {b: (mean_std(xy[b]), mean_std(yaw[b])) for b in BINS}
    print(f"{DISPLAY[robot]}")
    for b in BINS:
        xy_ms, yaw_ms = stats[robot][b]
        if xy_ms is None:
            print(f"  {b}: --")
            continue
        print(
            f"  {b}:  |xy| {xy_ms[0]:.3f} ± {xy_ms[1]:.3f}   "
            f"|yaw| {yaw_ms[0]:.3f} ± {yaw_ms[1]:.3f}"
        )
    print()


def shade(mean_xy):
    if mean_xy < 0.04:
        return "green!20"
    if mean_xy < 0.06:
        return "orange!25"
    return "red!20"


def pm(ms):
    m, s = ms
    return f"${m:.3f} \\pm {s:.3f}$"


def cell(xy_ms, yaw_ms):
    if xy_ms is None:
        return r"\cellcolor{gray!10} --"
    inner = (
        r"\begin{tabular}{@{}c@{}}"
        + pm(xy_ms)
        + r" \\ "
        + pm(yaw_ms)
        + r"\end{tabular}"
    )
    return rf"\cellcolor{{{shade(xy_ms[0])}}} {inner}"


def hdr_tex(label):
    a, b = label.split("~")
    return f"${a}\\sim{b}$"


n_col = len(BINS)
hdr = " & ".join(hdr_tex(b) for b in BINS)
body_lines = []
for robot in ROBOTS:
    cells = " & ".join(cell(*stats[robot][b]) for b in BINS)
    body_lines.append(DISPLAY[robot] + " & " + cells + r" \\")
body = "\n".join(body_lines)

latex = (
    r"% \usepackage{booktabs}" + "\n"
    r"% \usepackage[table]{xcolor}" + "\n"
    r"\begin{table*}[t]" + "\n"
    r"\centering" + "\n"
    r"\caption{Foot landing error vs.\ commanded step height $z$. "
    r"Each cell is mean $\pm$ std of planar foothold error $|xy|$ (m, top) "
    r"and yaw error $|yaw|$ (rad, bottom). Bins of $0.05$\,m.}" + "\n"
    r"\label{tab:z-cmd-error}" + "\n"
    r"\setlength{\tabcolsep}{3.5pt}" + "\n"
    r"\begin{tabular}{l*{" + str(n_col) + r"}{c}}" + "\n"
    r"\toprule" + "\n"
    r"& \multicolumn{" + str(n_col) + r"}{c}{\textbf{cmd$_z$ (m)}} \\" + "\n"
    r"\cmidrule(lr){2-" + str(n_col + 1) + r"}" + "\n"
    "& " + hdr + r" \\" + "\n"
    r"\midrule" + "\n"
    + body + "\n"
    r"\bottomrule" + "\n"
    r"\end{tabular}" + "\n"
    r"\end{table*}" + "\n"
)
print(latex)

G1
  -0.15~-0.10: --
  -0.10~-0.05: --
  -0.05~0.00: --
  0.00~0.05: --
  0.05~0.10: --
  0.10~0.15: --
  0.15~0.20: --

T1
  -0.15~-0.10: --
  -0.10~-0.05: --
  -0.05~0.00: --
  0.00~0.05: --
  0.05~0.10: --
  0.10~0.15: --
  0.15~0.20: --

TOCABI
  -0.15~-0.10: --
  -0.10~-0.05: --
  -0.05~0.00: --
  0.00~0.05: --
  0.05~0.10: --
  0.10~0.15: --
  0.15~0.20: --

KAPEX
  -0.15~-0.10: --
  -0.10~-0.05: --
  -0.05~0.00: --
  0.00~0.05: --
  0.05~0.10: --
  0.10~0.15: --
  0.15~0.20: --

% \usepackage{booktabs}
% \usepackage[table]{xcolor}
\begin{table*}[t]
\centering
\caption{Foot landing error vs.\ commanded step height $z$. Each cell is mean $\pm$ std of planar foothold error $|xy|$ (m, top) and yaw error $|yaw|$ (rad, bottom). Bins of $0.05$\,m.}
\label{tab:z-cmd-error}
\setlength{\tabcolsep}{3.5pt}
\begin{tabular}{l*{7}{c}}
\toprule
& \multicolumn{7}{c}{\textbf{cmd$_z$ (m)}} \\
\cmidrule(lr){2-8}
& $-0.15\sim-0.10$ & $-0.10\sim-0.05$ & $-0.05\sim0.00$ & $0.00\sim0.05$ & $0.05\sim0.1